<a href="https://colab.research.google.com/github/likith1525/ExcelR-data-science-assignments/blob/main/Codes/16Recommendation_System.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Anime Recommendation**

In [ ]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler

In [ ]:
anime = pd.read_csv('anime.csv')
anime.head()

,anime_id,name,genre,type,episodes,rating,members
0,32281,Kimi no Na wa.,"Drama, Romance, School, Supernatural",Movie,1,9.37,200630
1,5114,Fullmetal Alchemist: Brotherhood,"Action, Adventure, Drama, Fantasy, Magic, Mili...",TV,64,9.26,793665
2,28977,Gintama°,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51,9.25,114262
3,9253,Steins;Gate,"Sci-Fi, Thriller",TV,24,9.17,673572
4,9969,Gintama&#039;,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51,9.16,151266


In [ ]:
anime.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12294 entries, 0 to 12293
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   anime_id  12294 non-null  int64  
 1   name      12294 non-null  object 
 2   genre     12232 non-null  object 
 3   type      12269 non-null  object 
 4   episodes  12294 non-null  object 
 5   rating    12064 non-null  float64
 6   members   12294 non-null  int64  
dtypes: float64(1), int64(2), object(4)
memory usage: 672.5+ KB


In [ ]:
anime.shape

(12294, 7)

In [ ]:
anime.describe()

,anime_id,rating,members
count,12294.000000,12064.000000,1.229400e+04
mean,14058.221653,6.473902,1.807134e+04
std,11455.294701,1.026746,5.482068e+04
min,1.000000,1.670000,5.000000e+00
25%,3484.250000,5.880000,2.250000e+02
50%,10260.500000,6.570000,1.550000e+03
75%,24794.500000,7.180000,9.437000e+03
max,34527.000000,10.000000,1.013917e+06


In [ ]:
anime.isnull().sum()

,0
anime_id,0
name,0
genre,62
type,25
episodes,0
rating,230
members,0


In [ ]:
# Handling missing values
anime['genre'] = anime['genre'].fillna('')
anime['type'] = anime['type'].fillna('Unknown')

anime['rating'] = anime['rating'].fillna(anime['rating'].mean())

anime['episodes'] = anime['episodes'].replace('Unknown', np.nan)
anime['episodes'] = anime['episodes'].fillna(0)

anime['members'] = anime['members'].fillna(0)

In [ ]:
anime.isnull().sum()

,0
anime_id,0
name,0
genre,0
type,0
episodes,0
rating,0
members,0


## genre vectorization using TF-IDF

In [ ]:
tfidf = TfidfVectorizer(stop_words='english')

genre_matrix = tfidf.fit_transform(anime['genre'])

## Normalize The Numerical Features (Using MIn-Max Features)

In [ ]:
scaler = MinMaxScaler()

numeric_features =scaler.fit_transform(anime[["rating","episodes","members"]])

In [ ]:
#combine features
from scipy.sparse import hstack

combined_features = hstack((genre_matrix, numeric_features))

In [ ]:
#Index Mapping
indices= pd.Series(anime.index, index=anime['name']).drop_duplicates()

## Cosine Similarity
* Cosine similarity is used to measure the simiarity between the two vectors

In [ ]:
#COSINE Matrix
cosine_sim = cosine_similarity(combined_features)

## Recommendation Function

In [ ]:
def recommend_anime(title, cosine_sim=cosine_sim):

    idx = indices[title]

    similarity_scores = list(enumerate(cosine_sim[idx]))

    similarity_scores = sorted(
        similarity_scores,
        key=lambda x: x[1],
        reverse=True
    )

    similarity_scores = similarity_scores[1:11]

    anime_indices = [i[0] for i in similarity_scores]

    return anime[['name', 'genre', 'rating']].iloc[anime_indices]

In [ ]:
# Example recommendation
recommend_anime("Naruto")

,name,genre,rating
615,Naruto: Shippuuden,"Action, Comedy, Martial Arts, Shounen, Super P...",7.94
206,Dragon Ball Z,"Action, Adventure, Comedy, Fantasy, Martial Ar...",8.32
346,Dragon Ball,"Adventure, Comedy, Fantasy, Martial Arts, Shou...",8.16
1472,Naruto: Shippuuden Movie 4 - The Lost Tower,"Action, Comedy, Martial Arts, Shounen, Super P...",7.53
1573,Naruto: Shippuuden Movie 3 - Hi no Ishi wo Tsu...,"Action, Comedy, Martial Arts, Shounen, Super P...",7.50
486,Boruto: Naruto the Movie,"Action, Comedy, Martial Arts, Shounen, Super P...",8.03
1343,Naruto x UT,"Action, Comedy, Martial Arts, Shounen, Super P...",7.58
2997,Naruto Soyokazeden Movie: Naruto to Mashin to ...,"Action, Comedy, Martial Arts, Shounen, Super P...",7.11
588,Dragon Ball Kai,"Action, Adventure, Comedy, Fantasy, Martial Ar...",7.95
1103,Boruto: Naruto the Movie - Naruto ga Hokage ni...,"Action, Comedy, Martial Arts, Shounen, Super P...",7.68


## Threshold - based Recommendation

In [ ]:
def recommend_with_threshold(title, threshold=0.5):

    idx = indices[title]

    sim_scores = list(enumerate(cosine_sim[idx]))

    filtered = [
        (i, score)
        for i, score in sim_scores
        if score > threshold
    ]

    filtered = sorted(
        filtered,
        key=lambda x: x[1],
        reverse=True
    )

    anime_indices = [i[0] for i in filtered[1:11]]

    return anime[['name', 'rating']].iloc[anime_indices]

### Performance Analysis
* Advantages:
  * Easy to implement.
  * Fast recommendation generation.
  * Works well for Genre-Based similarity.
  * No user history is required.
* Limitations:
  * Cannot recommend new genres effectively.
  * Depends heavily on selected features.
  * Ignores user preferences.
  * Cold start problem exists.

### Area of Improvement
* Future improvements include:
  * Hybrid recommendation systems.
  * Deep learning recommendation models.
  * User based collaborative filtering.
  * Item based collaborative filtering.
  * Neural recommendation system.
  * Real-time recommendation updates.


###  Q1. Differences Between User-Based & Item-Based Collaborative Filtering
|User-Based Filtering|Item-Based Filtering|
|--------------------|--------------------|
|Recommends item based on similar users|Recommends item based on similar items|
|Finds users with similar kind of intrest|Finds item with similar characteristics|
|Computationally expensive|Faster and Scalable|
|EX: Users like Naruto & Bleach|EX: Naruto similar to Bleach|

### Q2. Collaborative Filtering
Collaborative Filtering is a recommendation technique that predicts user intrests using preferences of many users.
### working:
1. Collect User-Item interactions.
2. Find Similarities.
3. Predict preferences.
4. Recommend Items.
### Types:
* User-Based Collaborative filtering.
* Item-Based Collaborative filtering.
### Advantages:
* Personalized Recommendations.
* NO need for item features.
### Disadvantages:
* Cold start problem.
* Sparse data issue.
